In [1]:
import enum
import cv2

def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value) [1])


In [2]:
import traitlets
import ipywidgets.widgets as widgets
import time

import threading
import inspect
import ctypes

origin_widget = widgets.Image(format='jpeg', width=320, height=320)
mask_widget = widgets.Image(format='jpeg', width=320, height=320)
result_widget = widgets.Image(format='jpeg', width=320, height=320)

image_container = widgets.HBox([origin_widget, mask_widget, result_widget])
display(image_container)

In [3]:
import cv2
import numpy as np

img = cv2.imread('Wuerfel.jpg', 1)
#img = cv2.imread('Erwin482x482.jpg', 1)
#img = cv2.imread('image001.jpg', 1)

origin_widget.value = bgr8_to_jpeg(img)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

color_lower = np.array([0,143,163])   #rot
color_upper = np.array([11,255,255])
color_lower = np.array([55,80,66])   #green
color_upper = np.array([78,255,255])
#color_lower = np.array([78,43,46])   #blue
#color_upper = np.array([115,255,255])
#color_lower = np.array([26,100,91])   #yellow
#color_upper = np.array([32,255,255])

mask = cv2.inRange(hsv, color_lower, color_upper)
mask = cv2.erode(mask, None, iterations=2)  # Rauschen entfernen, None=3x3 Quadrat
mask = cv2.dilate(mask, None, iterations=2)
mask = cv2.GaussianBlur(mask, (3, 3), 0)    # Weichzeichnen

# cv2.RETR_EXTERNAL: Dieser Modus sagt der Funktion: „Suche nur die äußersten Konturen.“ Wenn du also einen weißen Ring hast, wird nur der äußere Rand erkannt, das Loch in der Mitte wird ignoriert.
# cv2.CHAIN_APPROX_SIMPLE: Dies spart Speicherplatz. Statt jeden einzelnen Pixel einer geraden Linie zu speichern, werden nur die Eckpunkte gespeichert (z. B. nur die 4 Ecken eines Rechtecks statt hunderter Punkte entlang der Seiten).

cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
cnt = max(cnts, key = cv2.contourArea)

# (x, y) (Output): Die Koordinaten des Kreismittelpunkts. Wichtig: Diese Werte sind Fließkommazahlen (Floats), keine Ganzzahlen.
# radius (Output): Der Radius des Kreises als Fließkommazahl.
(color_x, color_y), color_radius = cv2.minEnclosingCircle(cnt)
print(int(color_radius))
print(int(color_x), int(color_y))

cv2.circle(mask, (int(color_x), int(color_y)), int(color_radius), (255,0,255), 1)
 
cv2.drawContours(img, cnts, -1, (0, 255, 0), 2)

mask_widget.value = bgr8_to_jpeg(mask)

res = cv2.bitwise_and(img, img, mask=mask)
result_widget.value = bgr8_to_jpeg(res)


61
162 66
